# 🚀 학생 건강상태 분류 — 최종 스퍼트 (Lv5)

**현재 0.94864 → 목표 0.952+ (1등 0.95198)** · 지표: **balanced accuracy 확정** → `class_weight='balanced'` 유지가 정답.

> 이 구간(+0.003)은 "새 알고리즘 하나"로 메우는 게 아니라 **작은 이득을 5~6개 쌓아서** 메웁니다.
> 이 노트북의 무기 (기대 효과 큰 순):
>
> | # | 무기 | 기대 효과 | 비용 |
> |---|---|---|---|
> | 1 | **원본 데이터 결합** | +0.001~0.005 | Data 탭에서 링크 확인만 |
> | 2 | **스태킹 (메타모델)** | +0.0005~0.002 | 거의 공짜 (OOF 재활용) |
> | 3 | **Optuna 튜닝** | +0.001~0.003 | 시간 (GPU면 짧음) |
> | 4 | **seed averaging** | +0.0003~0.001 | 학습 ×seed 수 |
> | 5 | **10-fold 최종 학습** | +0.0002~0.001 | 학습 2배 |
> | 6 | **클래스별 확률 보정** | +0.0003~0.002 | 공짜 (balanced acc 전용) |
>
> 그리고 **속도 문제(Lv3 오래 걸림)의 해답 2개**가 함께 들어 있습니다:
> - **OOF 캐싱**: 한 번 학습한 모델의 OOF/test 확률을 디스크에 저장 → 앙상블 실험은 재학습 없이 몇 초
> - **GPU**: XGBoost·CatBoost는 GPU로 수 분 안에 끝남 (런타임 유형 → **T4 GPU** 선택하세요!)

### ⚠️ 시작 전 딱 하나 할 일
대회 **[Data 탭](https://www.kaggle.com/competitions/playground-series-s6e7/data)** 맨 아래 설명을 보세요.
Playground 대회는 보통 *"generated from a deep learning model trained on the ○○○ dataset"* 라며 **원본 데이터셋 링크**를 줍니다.
그 링크의 주소가 `kaggle.com/datasets/누구/이름` 이면 아래 설정의 `ORIGINAL_SLUG`에 `'누구/이름'`을 넣으세요. (없으면 빈 문자열 그대로 → 자동 스킵)


## 0. 설치 & 설정

**실행 전략 (3단계):**
1. `RUN_FRACTION=0.3, SEEDS=[42], RUN_OPTUNA=False` → 전체 흐름 검증 (빠름)
2. `RUN_FRACTION=1.0` → 베이스 앙상블 제출, 리더보드 확인
3. `RUN_OPTUNA=True, SEEDS=[42,2026,7], N_SPLITS=10` → 최종 풀파워 (GPU 필수)


In [ ]:
!pip install -q lightgbm xgboost catboost optuna kagglehub
print('설치 완료')

In [ ]:
import time, warnings, os, json, hashlib
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

# ═══════════ 핵심 설정 ═══════════
RUN_FRACTION     = 0.3      # ① 0.3으로 검증 → ② 최종 1.0
N_SPLITS         = 5        # 실험 5 → ③ 최종 제출 때 10
SEEDS            = [42]     # ③ 최종엔 [42, 2026, 7]  (seed averaging)
RUN_OPTUNA       = False    # ③ 튜닝 (GPU + 1~2시간 여유 있을 때 True)
OPTUNA_TRIALS    = 30
USE_CLASS_WEIGHT = True     # 지표 = balanced accuracy 확정 → True 고정!
ORIGINAL_SLUG    = ''       # ⭐ Data 탭에서 원본 데이터셋 확인 → 'owner/dataset-name'
# ═════════════════════════════════

RS = 42; np.random.seed(RS)
score_fn = balanced_accuracy_score
HAS_GPU = (os.system('nvidia-smi > /dev/null 2>&1') == 0)
print('GPU:', '사용 가능 ✅' if HAS_GPU else '❌ 없음 → 상단 메뉴 [런타임]-[런타임 유형 변경]-[T4 GPU] 강력 권장')

## 1. 데이터 로드 + 파생변수 (Lv4와 동일, self-contained)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_PATH='/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/dataset/kaggle_student_classification/'
train=pd.read_csv(DATA_PATH+'train.csv'); test=pd.read_csv(DATA_PATH+'test.csv')
TARGET,ID='health_condition','id'
numeric_features=['sleep_duration','heart_rate','bmi','calorie_expenditure','step_count','exercise_duration','water_intake']
categorical_features=['diet_type','stress_level','sleep_quality','physical_activity_level','smoking_alcohol','gender']
feature_cols=numeric_features+categorical_features
HR_HI=train['heart_rate'].quantile(0.75); STEP_LO=train['step_count'].quantile(0.25); WATER_LO=train['water_intake'].quantile(0.25)
ord_maps={'stress_level':{'low':0,'medium':1,'high':2},'sleep_quality':{'poor':0,'average':1,'good':2},
 'physical_activity_level':{'sedentary':0,'moderate':1,'active':2},'smoking_alcohol':{'no':0,'occasional':1,'yes':2}}
# 명목형 코드 매핑 (train 기준 고정 → test/원본에도 동일 적용)
code_maps={c:{k:i for i,k in enumerate(train[c].astype('category').cat.categories)} for c in ['diet_type','gender']}

def make_features(df):
    X=df.copy()
    for col,m in ord_maps.items(): X[col+'_ord']=X[col].map(m)
    X['sleep_debt']=(7-X['sleep_duration']).clip(lower=0); X['sleep_excess']=(X['sleep_duration']-9).clip(lower=0)
    X['sleep_ideal']=X['sleep_duration'].between(7,9).astype('float'); X.loc[X['sleep_duration'].isna(),'sleep_ideal']=np.nan
    X['bmi_cat']=pd.cut(X['bmi'],bins=[-np.inf,18.5,25,30,np.inf],labels=[0,1,2,3]).astype('float')
    X['bmi_abnormal']=(~X['bmi'].between(18.5,25)).astype('float'); X.loc[X['bmi'].isna(),'bmi_abnormal']=np.nan
    X['hr_high']=(X['heart_rate']>HR_HI).astype('float'); X.loc[X['heart_rate'].isna(),'hr_high']=np.nan
    X['steps_per_ex_min']=X['step_count']/(X['exercise_duration']+1); X['cal_per_step']=X['calorie_expenditure']/(X['step_count']+1)
    X['low_activity']=(X['step_count']<STEP_LO).astype('float'); X.loc[X['step_count'].isna(),'low_activity']=np.nan
    X['low_water']=(X['water_intake']<WATER_LO).astype('float'); X.loc[X['water_intake'].isna(),'low_water']=np.nan
    risk=pd.DataFrame(index=X.index)
    risk['a']=(X['sleep_duration']<6).astype(float); risk['b']=(X['stress_level']=='high').astype(float)
    risk['c']=(X['sleep_quality']=='poor').astype(float); risk['d']=(X['physical_activity_level']=='sedentary').astype(float)
    risk['e']=(X['step_count']<STEP_LO).astype(float); risk['f']=(X['smoking_alcohol']=='yes').astype(float)
    risk['g']=(~X['bmi'].between(18.5,25)).astype(float)
    X['lifestyle_risk_score']=risk.sum(axis=1); X['n_missing']=df[feature_cols].isnull().sum(axis=1)
    for c in feature_cols: X[c+'_isna']=df[c].isnull().astype('int8')
    for c,mp in code_maps.items(): X[c+'_code']=X[c].map(mp)
    return X

train_fe=make_features(train); test_fe=make_features(test)
print('train_fe', train_fe.shape, '| test_fe', test_fe.shape)

In [ ]:
# 변수 세트 — Lv4 진단(PRUNED vs FULL)에서 이긴 쪽으로 지정하세요. 기본 = FULL
FEATURES_PRUNED = numeric_features + [
    'lifestyle_risk_score','sleep_debt','sleep_ideal','bmi_cat','cal_per_step','steps_per_ex_min','low_activity',
    'stress_level_ord','sleep_quality_ord','physical_activity_level_ord','smoking_alcohol_ord',
    'diet_type_code','gender_code']
exclude=[ID,TARGET]+categorical_features
FEATURES_FULL=[c for c in train_fe.columns if c not in exclude and train_fe[c].dtype!='object']
BEST_FEATURES = FEATURES_FULL          # ← Lv4에서 PRUNED가 이겼다면 FEATURES_PRUNED로 변경
print('사용 변수:', len(BEST_FEATURES), '개')

le=LabelEncoder().fit(train_fe[TARGET]); CLASSES=le.classes_
if RUN_FRACTION<1.0:
    run=train_fe.groupby(TARGET,group_keys=False).apply(lambda s: s.sample(frac=RUN_FRACTION,random_state=RS))
else:
    run=train_fe
y_run=le.transform(run[TARGET])
print('실행 데이터:', run.shape, '| 클래스:', dict(zip(CLASSES, np.bincount(y_run))))

## 2. ⭐ 원본 데이터 결합 — Playground 대회 최대 치트키

Playground 대회의 train/test는 **실제 원본 데이터셋으로 학습한 생성모델**이 만든 합성 데이터입니다.
그래서 **원본을 train에 추가**하면 = 진짜 분포에서 나온 표본이 늘어나는 것 → 상위권이 거의 다 쓰는 기법.

**중요한 원칙 (누수 방지):** 원본은 **각 fold의 학습 부분에만** 추가합니다.
OOF 점수는 **대회 데이터로만** 계산해야 리더보드와 같은 잣대가 됩니다. (아래 엔진에 이미 반영됨)


In [ ]:
orig_fe, y_orig = None, None
if ORIGINAL_SLUG:
    try:
        import kagglehub
        p = kagglehub.dataset_download(ORIGINAL_SLUG)
        csvs=[os.path.join(r,f) for r,_,fs in os.walk(p) for f in fs if f.endswith('.csv')]
        print('발견된 CSV:', [os.path.basename(c) for c in csvs])
        orig = pd.read_csv(csvs[0])
        print('원본 shape:', orig.shape, '\n컬럼:', list(orig.columns))

        # 컬럼명이 다르면 여기서 매핑하세요. 예: orig = orig.rename(columns={'Sleep Duration':'sleep_duration'})
        RENAME = {}
        orig = orig.rename(columns=RENAME)

        need = set(feature_cols+[TARGET])
        if need.issubset(orig.columns) and set(orig[TARGET].dropna().unique()).issubset(set(CLASSES)):
            orig = orig.drop_duplicates(subset=feature_cols)
            orig_fe = make_features(orig)
            y_orig = le.transform(orig_fe[TARGET])
            print(f'✅ 원본 {len(orig_fe)}행 결합 준비 완료 (fold 학습부분에만 추가됨)')
        else:
            print('⚠️ 컬럼/라벨 불일치 → RENAME 딕셔너리로 컬럼명을 맞춰주세요. 부족한 컬럼:', need-set(orig.columns))
    except Exception as e:
        print('원본 로드 실패 → 스킵:', str(e)[:120])
else:
    print('ORIGINAL_SLUG 미지정 → 원본 결합 스킵 (Data 탭에서 꼭 확인해 보세요! 가장 싼 +0.001~0.005 입니다)')

## 3. OOF 엔진 v2 — 캐싱 + GPU + 원본증강 + 파라미터 주입

Lv4 엔진의 업그레이드판. 핵심 변화:

- **캐싱**: (모델, seed, fold수, fraction, 변수, 파라미터)가 같으면 저장된 확률을 즉시 로드 → **"Lv3처럼 오래 걸리는" 문제의 근본 해결.** 앙상블/스태킹/보정 실험은 재학습 없이 몇 초.
- **GPU**: XGBoost(`device='cuda'`)·CatBoost(`task_type='GPU'`)는 GPU에서 학습. (LightGBM·HistGB는 CPU가 안정적)
- **원본증강**: `orig_fe`가 있으면 각 fold의 **학습 부분에만** concat.
- **파라미터 주입**: Optuna가 찾은 best params를 그대로 넣을 수 있음.


In [ ]:
CACHE = DATA_PATH + 'oof_cache/'; os.makedirs(CACHE, exist_ok=True)

def make_model(name, seed, params=None):
    p = params or {}
    if name=='LightGBM':
        from lightgbm import LGBMClassifier
        base=dict(n_estimators=3000, learning_rate=0.03, num_leaves=63, subsample=0.8,
                  colsample_bytree=0.8, reg_lambda=1.0, n_jobs=-1, verbose=-1)
        base.update(p); return LGBMClassifier(random_state=seed, **base)
    if name=='XGBoost':
        from xgboost import XGBClassifier
        base=dict(n_estimators=3000, learning_rate=0.03, max_depth=6, subsample=0.8,
                  colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist',
                  early_stopping_rounds=100, eval_metric='mlogloss', n_jobs=-1, verbosity=0)
        if HAS_GPU: base['device']='cuda'
        base.update(p); return XGBClassifier(random_state=seed, **base)
    if name=='CatBoost':
        from catboost import CatBoostClassifier
        base=dict(iterations=3000, learning_rate=0.03, depth=6, l2_leaf_reg=3.0, verbose=0)
        if HAS_GPU: base['task_type']='GPU'
        base.update(p); return CatBoostClassifier(random_state=seed, **base)
    if name=='HistGB':
        from sklearn.ensemble import HistGradientBoostingClassifier
        base=dict(max_iter=3000, learning_rate=0.03, max_leaf_nodes=63, l2_regularization=1.0,
                  early_stopping=True, validation_fraction=0.1, n_iter_no_change=100)
        base.update(p); return HistGradientBoostingClassifier(random_state=seed, **base)

def fit_one(name, model, Xtr, ytr, Xva, yva, sw):
    if name=='LightGBM':
        from lightgbm import early_stopping, log_evaluation
        model.fit(Xtr,ytr,sample_weight=sw,eval_set=[(Xva,yva)],
                  callbacks=[early_stopping(100,verbose=False),log_evaluation(0)])
    elif name=='XGBoost':
        model.fit(Xtr,ytr,sample_weight=sw,eval_set=[(Xva,yva)],verbose=False)
    elif name=='CatBoost':
        model.fit(Xtr,ytr,sample_weight=sw,eval_set=(Xva,yva),early_stopping_rounds=100,verbose=0)
    else:
        model.fit(Xtr,ytr,sample_weight=sw)
    return model

def run_oof(name, features, seed=42, params=None):
    key = hashlib.md5(json.dumps(dict(n=name,f=features,s=seed,k=N_SPLITS,fr=RUN_FRACTION,
          cw=USE_CLASS_WEIGHT,orig=orig_fe is not None,p=params), sort_keys=True, default=str
          ).encode()).hexdigest()[:12]
    fpath = CACHE + f'{name}_s{seed}_{key}.npz'
    if os.path.exists(fpath):                     # ← 캐시 히트: 재학습 없음!
        z=np.load(fpath); sc=score_fn(y_run, z['oof'].argmax(1))
        print(f'  {name:9s} seed{seed} 💾캐시  OOF={sc:.5f}'); return z['oof'], z['test'], sc
    X=run[features]; Xtest=test_fe[features]; n_cls=len(CLASSES)
    oof=np.zeros((len(X),n_cls)); test_proba=np.zeros((len(Xtest),n_cls))
    skf=StratifiedKFold(N_SPLITS,shuffle=True,random_state=seed); t0=time.time()
    for tr,va in skf.split(X,y_run):
        Xtr,ytr=X.iloc[tr],y_run[tr]; Xva,yva=X.iloc[va],y_run[va]
        if orig_fe is not None:                   # 원본은 학습부분에만 (검증은 대회 데이터만!)
            Xtr=pd.concat([Xtr,orig_fe[features]]); ytr=np.concatenate([ytr,y_orig])
        sw=compute_sample_weight('balanced',ytr) if USE_CLASS_WEIGHT else None
        m=fit_one(name,make_model(name,seed,params),Xtr,ytr,Xva,yva,sw)
        oof[va]=m.predict_proba(Xva); test_proba+=m.predict_proba(Xtest)/N_SPLITS
    sc=score_fn(y_run,oof.argmax(1))
    np.savez_compressed(fpath,oof=oof,test=test_proba)
    print(f'  {name:9s} seed{seed}  OOF={sc:.5f}  ({time.time()-t0:.0f}s, 캐시 저장)')
    return oof,test_proba,sc
print('OOF 엔진 v2 준비 완료 ✅')

## 4. (선택) Optuna 튜닝 — 부스팅을 제대로 짜내기

Lv3의 RandomizedSearch(15회)는 맛보기였어요. Optuna는 **이전 시도 결과를 보고 다음 후보를 똑똑하게** 고릅니다(TPE).
- 빠르게 하려고 **80/20 holdout + early stopping**으로 평가 (fold CV보다 5배 빠르고, 파라미터 우열 판단에는 충분)
- 결과는 JSON으로 저장 → 다음 실행 땐 자동 재사용
- `RUN_OPTUNA=False`면 이 셀은 저장된 파라미터만 불러옵니다


In [ ]:
PARAM_FILE = DATA_PATH + 'best_params.json'
BEST_PARAMS = json.load(open(PARAM_FILE)) if os.path.exists(PARAM_FILE) else {}

if RUN_OPTUNA:
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
    Xh, Xv, yh, yv = train_test_split(run[BEST_FEATURES], y_run, test_size=0.2,
                                      stratify=y_run, random_state=RS)
    if orig_fe is not None:
        Xh=pd.concat([Xh,orig_fe[BEST_FEATURES]]); yh=np.concatenate([yh,y_orig])
    swh=compute_sample_weight('balanced',yh) if USE_CLASS_WEIGHT else None

    def space(name,t):
        if name=='LightGBM': return dict(
            learning_rate=t.suggest_float('learning_rate',0.01,0.1,log=True),
            num_leaves=t.suggest_int('num_leaves',31,255),
            min_child_samples=t.suggest_int('min_child_samples',10,100),
            subsample=t.suggest_float('subsample',0.6,1.0),
            colsample_bytree=t.suggest_float('colsample_bytree',0.5,1.0),
            reg_lambda=t.suggest_float('reg_lambda',0.01,10,log=True))
        if name=='XGBoost': return dict(
            learning_rate=t.suggest_float('learning_rate',0.01,0.1,log=True),
            max_depth=t.suggest_int('max_depth',4,10),
            min_child_weight=t.suggest_int('min_child_weight',1,20),
            subsample=t.suggest_float('subsample',0.6,1.0),
            colsample_bytree=t.suggest_float('colsample_bytree',0.5,1.0),
            reg_lambda=t.suggest_float('reg_lambda',0.01,10,log=True))
        if name=='CatBoost': return dict(
            learning_rate=t.suggest_float('learning_rate',0.01,0.1,log=True),
            depth=t.suggest_int('depth',4,9),
            l2_leaf_reg=t.suggest_float('l2_leaf_reg',0.5,20,log=True))

    for name in ['LightGBM','XGBoost','CatBoost']:
        def objective(t):
            m=fit_one(name,make_model(name,RS,space(name,t)),Xh,yh,Xv,yv,swh)
            return score_fn(yv,m.predict_proba(Xv).argmax(1))
        st=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=RS))
        t0=time.time(); st.optimize(objective,n_trials=OPTUNA_TRIALS,show_progress_bar=True)
        BEST_PARAMS[name]=st.best_params
        print(f'{name}: holdout={st.best_value:.5f} ({time.time()-t0:.0f}s)\n  {st.best_params}')
    json.dump(BEST_PARAMS,open(PARAM_FILE,'w'),indent=1)
    print('저장:',PARAM_FILE)
else:
    print('RUN_OPTUNA=False | 저장된 파라미터:', list(BEST_PARAMS.keys()) or '없음(기본값 사용)')

## 5. 멀티시드 × 4모델 OOF 학습

**seed averaging의 원리**: 같은 모델도 seed(fold 분할·샘플링)가 다르면 **다르게 틀립니다**.
여러 seed의 확률을 평균하면 운에 의한 분산이 줄어요. 앙상블 재료가 4모델 → 4모델×3seed = 12개로 늘어나는 효과.

⏳ 캐시 덕분에 **이미 돌린 조합은 건너뜁니다.** 중간에 끊겨도 다시 실행하면 이어서 돌아요.


In [ ]:
MODELS=['LightGBM','XGBoost','CatBoost','HistGB']
runs={}   # (model, seed) -> (oof, test, score)
print(f'▶ {len(MODELS)}모델 × {len(SEEDS)}seed OOF 학습')
for name in MODELS:
    for seed in SEEDS:
        try: runs[(name,seed)]=run_oof(name,BEST_FEATURES,seed,BEST_PARAMS.get(name))
        except Exception as e: print(f'  {name} seed{seed} 실패: {str(e)[:80]}')

tbl=pd.DataFrame([{'model':n,'seed':s,'OOF_bal_acc':sc} for (n,s),(o,t,sc) in runs.items()])
display(tbl.sort_values('OOF_bal_acc',ascending=False).round(5))

## 6. ⭐ 앙상블 3단 비교 — 단순평균 vs 가중평균 vs 스태킹

**스태킹이 뭐가 다른가?** 단순/가중 평균은 "모델 신뢰도"를 사람이(또는 랜덤서치가) 정합니다.
스태킹은 4모델의 OOF 확률(12차원)을 **입력 변수**로 받는 작은 메타모델(로지스틱 회귀)을 **학습**시켜서,
"CatBoost가 unhealthy라고 하고 LGBM이 애매해하면 → unhealthy" 같은 **조건부 신뢰 규칙**까지 데이터가 배웁니다.

메타모델 점수도 **cross_val_predict**로 정직하게 측정합니다 (OOF 위에서 또 CV — 누수 방지 2중 장치).


In [ ]:
keys=list(runs.keys())
oof_stack=np.stack([runs[k][0] for k in keys])    # (n_runs, n_samples, 3)
test_stack=np.stack([runs[k][1] for k in keys])

# ① 단순 평균
oof_mean=oof_stack.mean(0); test_mean=test_stack.mean(0)
s_mean=score_fn(y_run,oof_mean.argmax(1))

# ② 가중 평균 (Dirichlet 랜덤서치 4000회)
rng=np.random.RandomState(RS); best_w=np.ones(len(keys))/len(keys); s_w=s_mean
for _ in range(4000):
    w=rng.dirichlet(np.ones(len(keys)))
    s=score_fn(y_run,np.tensordot(w,oof_stack,axes=(0,0)).argmax(1))
    if s>s_w: s_w,best_w=s,w
oof_wavg=np.tensordot(best_w,oof_stack,axes=(0,0)); test_wavg=np.tensordot(best_w,test_stack,axes=(0,0))

# ③ 스태킹 (메타 = 로지스틱 회귀)
meta_X=np.hstack([runs[k][0] for k in keys])      # (n_samples, n_runs*3)
meta_T=np.hstack([runs[k][1] for k in keys])
meta=LogisticRegression(max_iter=2000,class_weight='balanced' if USE_CLASS_WEIGHT else None)
oof_meta_pred=cross_val_predict(meta,meta_X,y_run,cv=StratifiedKFold(5,shuffle=True,random_state=RS),
                                method='predict_proba',n_jobs=-1)
s_stack=score_fn(y_run,oof_meta_pred.argmax(1))
meta.fit(meta_X,y_run); test_stackp=meta.predict_proba(meta_T)

cmp=pd.DataFrame({'전략':['① 단순평균','② 가중평균','③ 스태킹'],'OOF_bal_acc':[s_mean,s_w,s_stack]})
display(cmp.round(5))
best_i=int(np.argmax([s_mean,s_w,s_stack]))
FINAL_OOF=[oof_mean,oof_wavg,oof_meta_pred][best_i]
FINAL_TEST=[test_mean,test_wavg,test_stackp][best_i]
print(f'채택: {cmp["전략"][best_i]}  OOF={cmp["OOF_bal_acc"][best_i]:.5f}')
print('(참고: ②는 OOF에 과적합 여지가 있어, ①과 차이가 +0.0005 미만이면 ①이나 ③을 믿는 게 안전)')

## 7. balanced accuracy 전용 마무리 — 클래스별 확률 보정

**왜 되나?** balanced accuracy는 3개 클래스 recall의 평균 = **세 클래스가 똑같이 중요**합니다.
argmax는 "확률 그대로" 자르지만, 소수 클래스(fit·unhealthy)의 확률에 약간의 **배수(multiplier)** 를 곱해
경계선 샘플을 소수 클래스 쪽으로 조금 넘기면 macro recall이 오르는 경우가 많아요.
OOF에서 최적 배수를 찾아 test에 그대로 적용합니다. (class_weight와 별개로 추가되는 마지막 미세조정)


In [ ]:
def tune_multipliers(proba,y,iters=5000):
    rng=np.random.RandomState(RS); best=np.ones(proba.shape[1])
    best_s=score_fn(y,proba.argmax(1))
    for _ in range(iters):
        m=np.exp(rng.normal(0,0.15,proba.shape[1]))     # 1.0 근처 랜덤 배수
        s=score_fn(y,(proba*m).argmax(1))
        if s>best_s: best_s,best=s,m
    return best,best_s

base_s=score_fn(y_run,FINAL_OOF.argmax(1))
mult,cal_s=tune_multipliers(FINAL_OOF,y_run)
print(f'보정 전 OOF={base_s:.5f} → 보정 후 OOF={cal_s:.5f} (+{cal_s-base_s:.5f})')
print('클래스 배수:', dict(zip(CLASSES,mult.round(3))))
USE_CAL = (cal_s-base_s) > 0.0003          # 이득이 노이즈 수준이면 미적용 (OOF 과적합 방지)
print('적용 여부:', USE_CAL)

## 8. 제출 파일 생성

OOF 점수가 가장 좋은 조합으로 제출 파일을 만듭니다. 하루 제출 횟수가 제한이니
**OOF로 고르고, 최고 후보 1~2개만 제출**하는 습관이 중요합니다 (OOF↔리더보드가 같이 움직이는지도 확인).


In [ ]:
final_proba = FINAL_TEST*mult if USE_CAL else FINAL_TEST
pred=le.inverse_transform(final_proba.argmax(1))
tag=('full' if RUN_FRACTION>=1.0 else f'frac{RUN_FRACTION}')+f'_{len(SEEDS)}seed_{N_SPLITS}fold'
path=DATA_PATH+f'submission_lv5_{tag}.csv'
pd.DataFrame({ID:test[ID],TARGET:pred}).to_csv(path,index=False)
print('저장:',path)
print('예측 분포:',pd.Series(pred).value_counts(normalize=True).round(3).to_dict())
print(f'\n기대 리더보드 ≈ OOF {score_fn(y_run,(FINAL_OOF*mult if USE_CAL else FINAL_OOF).argmax(1)):.5f}')

## ✅ 실행 플랜 & 남은 카드

### 실행 순서 (이대로 하세요)
| 단계 | 설정 | 목적 | 예상 시간(T4 GPU) |
|---|---|---|---|
| ① | `RUN_FRACTION=0.3` | 파이프라인 검증 | ~10분 |
| ② | `RUN_FRACTION=1.0` | 베이스 제출 → 리더보드 확인 | ~30-60분 |
| ③ | `+ RUN_OPTUNA=True` | 튜닝 반영 재실행 (캐시 덕에 튜닝분만 추가) | +1-2시간 |
| ④ | `+ SEEDS=[42,2026,7], N_SPLITS=10` | 최종 풀파워 | 수 시간 (밤에 걸어두기) |

### 왜 이 노트북은 Lv3처럼 "매번" 오래 걸리지 않나
- **오래 걸리는 건 학습 한 번뿐**, 캐시(oof_cache/) 덕에 앙상블·스태킹·보정 실험은 몇 초.
- Lv3는 발표 요건용 비교표였고, 점수 생산은 부스팅 4종이면 충분. **학습시간 ≠ 점수**입니다.

### 그래도 부족하면 (마지막 카드)
1. **의사라벨링**: test 예측 중 확률 0.99+ 샘플을 train에 추가 → 재학습 (과적합 주의, 소량만)
2. **피처 상호작용 2차전**: `*_ord` 상위 변수들의 곱/합 조합 (`stress_ord × sleep_debt` 등)
3. **다른 결의 모델 추가**: 간단한 MLP 1개를 앙상블에 (부스팅과 다르게 틀림 → 평균의 힘↑)

### 정직한 한마디
0.94864→0.95198 구간의 마지막 0.001은 **운(seed)과 미세조정의 영역**이기도 합니다.
상위권도 마법이 아니라 위 카드를 전부 쌓은 것 + 제출 반복이에요. OOF 기준으로 냉정하게 고르고,
**리더보드에 과적합하지 않는 것**(public 점수 쫓아 이것저것 제출)이 오히려 최종 순위를 지킵니다.
